# Ноутбук 04. Бінарна класифікація шаблонів — навчання $f_B$

Цей ноутбук тренує бінарний детектор $f_B: \mathbb{R}^{768} \to \{0, 1\}$ безпосередньо на парах $(\beta(t), \ell(t))$, де $\beta(t)$ — BERT-вектор шаблону, а $\ell(t)$ — фінальна (з можливим експертним коригуванням) бінарна мітка з ноутбука 03. KMeans-кластери **не використовуються** як навчальний сигнал (амендмент архітектури — див. розділи 2.1, 2.3 тези).

Етапи:

1. Завантаження BERT-векторів `(77, 768)` та фінальних міток $\ell(t)$.
2. L2-нормалізація рядків (BERT анізотропний, лінійні класифікатори чутливі до масштабу).
3. Розбиття на train/test 80 % / 20 % зі стратифікацією за $\ell$ на рівні **шаблонів** (Перевірка 1 тези).
4. Навчання GaussianNB / LogisticRegression / LinearSVC з `sample_weight = support(t)` для компенсації дисбалансу.
5. Метрики: **MCC** (основна), F1-macro, ROC-AUC, час навчання та інференсу на запис (CPU, один потік).
6. Baseline: `DummyClassifier(strategy='most_frequent')` — нижня межа адекватності з тези 2.4.
7. Перевірка 2 (теза 2.3): MCC того самого передбачення проти rule-based міток $\ell^{(0)}$ — без переnaвчання.

Вихідний артефакт: `data/processed/classification/metrics_fb.csv`.

In [1]:
from __future__ import annotations

import os

# CPU single-thread for deterministic, reproducible timing. Must be set BEFORE
# numpy/sklearn import — BLAS reads these at C-extension load time.
for _var in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "BLIS_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[_var] = "1"

import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PROCESSED = ROOT / 'data' / 'processed'
EMBEDDINGS_DIR = DATA_PROCESSED / 'embeddings'
CLUSTERS_DIR = DATA_PROCESSED / 'clusters'
RESULTS_DIR = DATA_PROCESSED / 'classification'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from src.classification.trainer import (
    evaluate,
    fit_with_timing,
    inference_seconds_per_record,
    make_baseline,
    make_classifiers,
)
from src.io.persistence import load_json, load_numpy

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f'numpy = {np.__version__}')
print(f'pandas = {pd.__version__}')

numpy = 2.4.4
pandas = 3.0.2


## 1. Завантаження ембедингів, міток та мапінгу

Зчитуємо чотири артефакти попередніх ноутбуків:

* `embeddings/zookeeper_full_embeddings.npy` $(77, 768)$ `float32` — BERT [CLS]-вектори.
* `embeddings/zookeeper_full_id_mapping.json` — порядок `row_index → template_id`.
* `clusters/template_to_binary_label.json` — фінальна мітка $\ell(t)$ + `support(t)` + `level_counts`.
* `clusters/template_to_rule_label.json` — мінімальний контракт $\{\texttt{template\_id}, \texttt{rule\_label}\}$ для Перевірки 2.

Стикуємо їх через `template_id` так, щоб усі чотири масиви — `X`, `y`, `w`, `y_rule` — мали однаковий порядок рядків (тобто порядок з `id_mapping`).

In [2]:
embeddings = load_numpy(EMBEDDINGS_DIR / 'zookeeper_full_embeddings.npy')
id_mapping: list[dict] = load_json(EMBEDDINGS_DIR / 'zookeeper_full_id_mapping.json')
template_to_label = load_json(CLUSTERS_DIR / 'template_to_binary_label.json')
template_to_rule = load_json(CLUSTERS_DIR / 'template_to_rule_label.json')

# Lookup dicts keyed by template_id.
final_by_tid: dict[int, int] = {r['template_id']: r['final_label'] for r in template_to_label}
support_by_tid: dict[int, int] = {r['template_id']: r['support'] for r in template_to_label}
rule_by_tid: dict[int, int] = {r['template_id']: r['rule_label'] for r in template_to_rule}

# Every embedding row must have a label, and vice versa — on both files.
embed_tids = {entry['template_id'] for entry in id_mapping}
assert embed_tids == set(final_by_tid), \
    f'embedding/label template_id mismatch: {embed_tids ^ set(final_by_tid)}'
assert embed_tids == set(rule_by_tid), \
    f'embedding/rule  template_id mismatch: {embed_tids ^ set(rule_by_tid)}'

# Align everything to id_mapping row order.
# X shape (77, 768) float32; y/w/y_rule shape (77,).
X = embeddings.astype(np.float32)
y = np.array([final_by_tid[m['template_id']] for m in id_mapping], dtype=int)
w = np.array([support_by_tid[m['template_id']] for m in id_mapping], dtype=float)
y_rule = np.array([rule_by_tid[m['template_id']] for m in id_mapping], dtype=int)

assert X.shape == (77, 768), X.shape
assert y.shape == (77,) and w.shape == (77,) and y_rule.shape == (77,)
assert set(np.unique(y).tolist()) == {0, 1}, f'y must contain both classes, got {set(y.tolist())}'

print(f'X      = {X.shape} {X.dtype}')
print(f'y      = {y.shape} (binary)')
print(f'w      = {w.shape} (template support)')
print(f'y_rule = {y_rule.shape} (rule-only labels for Перевірка 2)')

X      = (77, 768) float32
y      = (77,) (binary)
w      = (77,) (template support)
y_rule = (77,) (rule-only labels for Перевірка 2)


## 2. L2-нормалізація рядків

Без нормалізації лінійні класифікатори (особливо LinearSVC та LogReg) перекошуються в напрямку шаблонів з більшою нормою BERT-вектора. У ноутбуку 03 §2 ми бачили, що норми лежать у вузькій смузі $[13.4, 15.8]$, але навіть таке коливання впливає на margin/гіперплощину. Тому переводимо всі рядки на одиничну гіперсферу (та сама нормалізація, що в діагностичному KMeans).

In [3]:
X_norm = normalize(X, norm='l2', axis=1).astype(np.float32)
# X_norm shape (77, 768) float32; row norms == 1 ± eps.

post_norms = np.linalg.norm(X_norm, axis=1)
assert np.allclose(post_norms, 1.0, atol=1e-5), 'L2 normalization did not produce unit vectors'

print(f'X_norm = {X_norm.shape} {X_norm.dtype}')
print(f'row-norm min/max = {post_norms.min():.6f} / {post_norms.max():.6f}')

X_norm = (77, 768) float32
row-norm min/max = 1.000000 / 1.000000


## 3. Баланс класів

На рівні **шаблонів** очікувано приблизно $16$ позитивних / $61$ негативний (у поточному стані без експертних корекцій — $18 / 59$). На рівні **записів** ситуація обернена: позитивний клас домінує (~$66\%$ трафіку — переважно WARN-шаблони). Саме тому `sample_weight = support(t)` — навчання зміщується на користь практично домінуючих подій без зміни постановки задачі (теза 2.1).

Тут же друкуємо повну (по всіх 77 шаблонах) кількість розбіжностей між експертною та rule-based розмітками — це базова цифра для Перевірки 2.

In [4]:
class_counts = Counter(y.tolist())
print(f'templates: y=0 -> {class_counts[0]}, y=1 -> {class_counts[1]} (of {y.size})')

# Weighted record-level distribution (sample_weight = support).
pos_records = int(w[y == 1].sum())
neg_records = int(w[y == 0].sum())
total_records = pos_records + neg_records
print(
    f'records (via support): y=0 -> {neg_records} '
    f'({100 * neg_records / total_records:.2f} %), '
    f'y=1 -> {pos_records} ({100 * pos_records / total_records:.2f} %)'
)

# Rule vs expert label disagreement on the FULL template set.
n_disagree_total = int((y != y_rule).sum())
print(f'rule vs expert disagreement (all templates): {n_disagree_total} of {y.size}')

templates: y=0 -> 59, y=1 -> 18 (of 77)
records (via support): y=0 -> 25256 (33.96 %), y=1 -> 49124 (66.04 %)
rule vs expert disagreement (all templates): 0 of 77


## 4. Train/test split на рівні шаблонів (Перевірка 1)

Стратифіковане розбиття $80\,\% / 20\,\%$ на $(X, y, w, y_{\text{rule}})$ з `random_state=42`. Розбиття саме **на рівні шаблонів** (не записів) запобігає витоку даних: різні записи одного шаблону не можуть одночасно потрапити у train та test.

За малої потужності позитивного класу (~$16$) тестова вибірка містить лише ~$3$–$4$ позитиви — друкуємо точні кількості та явно вказуємо caveat у разі $<3$ прикладів класу (теза 2.4).

In [5]:
X_train, X_test, y_train, y_test, w_train, w_test, y_rule_train, y_rule_test = train_test_split(
    X_norm, y, w, y_rule,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'train: X={X_train.shape}, y={y_train.shape}, balance = {dict(Counter(y_train.tolist()))}')
print(f'test : X={X_test.shape}, y={y_test.shape}, balance = {dict(Counter(y_test.tolist()))}')

test_class_counts = Counter(y_test.tolist())
for cls in (0, 1):
    n_cls = test_class_counts[cls]
    if n_cls < 3:
        print(
            f'⚠ caveat: class {cls} has only {n_cls} sample(s) in the test split; '
            'MCC and ROC-AUC are statistically unstable (теза 2.4).'
        )

n_disagree_test = int((y_test != y_rule_test).sum())
print(f'rule vs expert disagreement (test split): {n_disagree_test} of {y_test.size}')

train: X=(61, 768), y=(61,), balance = {1: 14, 0: 47}
test : X=(16, 768), y=(16,), balance = {0: 12, 1: 4}
rule vs expert disagreement (test split): 0 of 16


## 5. Навчання трьох класифікаторів + baseline

Для кожної моделі:

1. `fit(X_train, y_train, sample_weight=w_train)` — час `fit` записується.
2. `predict(X_test)` → `evaluate(...)` рахує MCC, F1-macro, ROC-AUC та `MCC_vs_rulebased` (Перевірка 2).
3. `inference_seconds_per_record(model, X_test, repeats=200)` — середній час `predict` на один запис.

`DummyClassifier(strategy='most_frequent')` навчається тими ж даними; його MCC має дорівнювати $0$ — це нижня межа адекватності (теза 2.4).

In [6]:
classifiers = make_classifiers(random_state=RANDOM_STATE)
classifiers['Baseline'] = make_baseline(random_state=RANDOM_STATE)

per_model: dict[str, dict[str, float]] = {}

for name, clf in classifiers.items():
    train_s = fit_with_timing(clf, X_train, y_train, sample_weight=w_train)
    metrics = evaluate(clf, X_test, y_test, y_rule_test)
    infer_s = inference_seconds_per_record(clf, X_test, repeats=200)
    metrics['train_s'] = train_s
    metrics['infer_s_per_rec'] = infer_s
    per_model[name] = metrics
    print(
        f'{name:<10}  MCC={metrics["MCC"]:+.4f}  '
        f'F1={metrics["F1_macro"]:.4f}  '
        f'ROC-AUC={metrics["ROC_AUC"]:.4f}  '
        f'MCC(rule)={metrics["MCC_vs_rulebased"]:+.4f}  '
        f'fit={train_s:.4g}s  infer={infer_s:.4g}s/rec'
    )

GaussianNB  MCC=+0.3637  F1=0.6537  ROC-AUC=0.7292  MCC(rule)=+0.3637  fit=0.0007523s  infer=3.253e-06s/rec
LogReg      MCC=+0.4623  F1=0.7257  ROC-AUC=0.9167  MCC(rule)=+0.4623  fit=0.004782s  infer=1.616e-06s/rec
LinearSVC   MCC=+0.6667  F1=0.8333  ROC-AUC=0.9375  MCC(rule)=+0.6667  fit=0.06269s  infer=1.575e-06s/rec
Baseline    MCC=+0.0000  F1=0.2000  ROC-AUC=0.5000  MCC(rule)=+0.0000  fit=9.013e-05s  infer=4.766e-06s/rec


## 6. Підсумкова таблиця та експорт

Збираємо результати у `DataFrame` з фіксованим порядком рядків (`GaussianNB → LogReg → LinearSVC → Baseline`) і колонок (`MCC, F1_macro, ROC_AUC, MCC_vs_rulebased, train_s, infer_s_per_rec`). Друкуємо з $4$ значущими цифрами; CSV зберігається з повною точністю.

**Інтерпретація Перевірки 2 (теза 2.3).** Якщо обидві колонки `MCC` та `MCC_vs_rulebased` високі, при цьому експертні та rule-based мітки розходяться хоча б на одному шаблоні тестового сплету, це доводить, що класифікатор уловлює семантику, яка не зводиться до правила за `Level`. У поточному стані `corrections` у ноутбуці 03 порожні — обидві мітки тотожні, тож `MCC == MCC_vs_rulebased`; стовпець потрібен як стабільна точка входу для подальших ітерацій з експертними правками.

In [7]:
row_order = ['GaussianNB', 'LogReg', 'LinearSVC', 'Baseline']
col_order = ['MCC', 'F1_macro', 'ROC_AUC', 'MCC_vs_rulebased', 'train_s', 'infer_s_per_rec']

metrics_df = pd.DataFrame.from_dict(per_model, orient='index').loc[row_order, col_order]

# 4 sig figs for printed view; CSV keeps full precision.
with pd.option_context('display.float_format', '{:.4g}'.format):
    print(metrics_df)

csv_path = RESULTS_DIR / 'metrics_fb.csv'
metrics_df.to_csv(csv_path, index_label='model')
print(f'\nsaved: {csv_path}  ({csv_path.stat().st_size} bytes)')

# Inline checks (теза 2.3 Перевірка 1 + sanity floor).
assert metrics_df.shape == (4, len(col_order)), metrics_df.shape
assert not metrics_df['MCC'].isna().any(), 'MCC column must not contain NaN'
assert abs(metrics_df.loc['Baseline', 'MCC']) < 1e-9, (
    f"baseline MCC expected 0 (constant prediction); got {metrics_df.loc['Baseline', 'MCC']}"
)
print('asserts passed.')

              MCC  F1_macro  ROC_AUC  MCC_vs_rulebased   train_s  \
GaussianNB 0.3637    0.6537   0.7292            0.3637 0.0007523   
LogReg     0.4623    0.7257   0.9167            0.4623  0.004782   
LinearSVC  0.6667    0.8333   0.9375            0.6667   0.06269   
Baseline        0       0.2      0.5                 0 9.013e-05   

            infer_s_per_rec  
GaussianNB        3.253e-06  
LogReg            1.616e-06  
LinearSVC         1.575e-06  
Baseline          4.766e-06  

saved: /Users/roman/Personal/dyploma/data/processed/classification/metrics_fb.csv  (512 bytes)
asserts passed.


## Висновки

Три класифікатори ($\text{GaussianNB}$, $\text{LogReg}$, $\text{LinearSVC}$) натреновано на template-level парах $(\beta(t), \ell(t))$ з `sample_weight = support(t)`. Baseline-передиктор більшості показав $\text{MCC} = 0$ — це підтверджує, що будь-який нетривіальний результат вищих моделей є реальним сигналом, а не артефактом дисбалансу. Cross-check MCC проти rule-based міток ($\ell^{(0)}$) обчислено для майбутніх ітерацій з експертними коригуваннями (теза 2.3 Перевірка 2).

Експорт: `data/processed/classification/metrics_fb.csv`. Наступний крок — якісний аналіз помилок з attention-картами BERT (Перевірка 3 тези 2.3, ноутбук 05).